# Mərhələ 6: Qabaqcıl Modelləşdirmə — Kaggle Nəticənizi Yaxşılaşdırmaq

## Məqsədlər

Bu mərhələdə siz:

1. Xətti Reqressiyanın Kaggle-da niyə cəmi 0.19 topladığını və nəyin səhv getdiyini anlayacaqsınız
2. RMSLE-nin nə olduğunu və hədəf dəyişənin loqarifmik transformasiyasının niyə kritik olduğunu öyrənəcəksiniz
3. Overfitting-i düzəltmək üçün reqlyarizasiyalı modelləri (Ridge, Lasso, ElasticNet) tətbiq edəcəksiniz
4. Qeyri-xətti qanunauyğunluqları tutan ağac əsaslı modelləri (Random Forest, XGBoost, LightGBM) öyrədəcəksiniz
5. Bütün modelləri düzgün çarpaz yoxlama (cross-validation) ilə müqayisə edəcəksiniz
6. Ən yüksək nəticə üçün ən yaxşı modelləri yekun ansamblda (ensemble) birləşdirəcəksiniz
7. Kaggle təqdimatı yaradıb yaxşılaşmanı ölçəcəksiniz

## Mərhələ 5-də Nəyi Qaçırdıq və Bu Niyə Vacibdir

Mərhələ 5-dəki Xətti Reqressiya tapşırığımızın bir neçə kritik məhdudiyyəti var idi:

- **SalePrice üzərində loqarifmik transformasiya yox idi**: Kaggle bu müsabiqəni RMSE ilə deyil, RMSLE (Root Mean Squared Log Error - Orta Kvadratik Loqarifmik Xətanın Kökü) ilə qiymətləndirir. Xam qiymətləri proqnozlaşdırdıqda böyük evlər xətanın böyük hissəsini təşkil edirdi. Loqarifmik məkanda işləmək RMSLE-ni sadə RMSE-yə çevirir və modellərimiz birbaşa onu optimallaşdırır.
- **Reqlyarizasiya yox idi**: One-hot kodlaşdırma və xüsusiyyət mühəndisliyindən sonra 200+ xüsusiyyətimiz var idi. Adi Xətti Reqressiya cəza tətbiq etmədən HAMISINA uyğunlaşır, bu da əmsalların partlamasına səbəb olur — klassik overfitting.
- **Yalnız bir model**: Xətti Reqressiya xüsusiyyətlərlə qiymət arasında düz xətt əlaqəsi olduğunu fərz edir. Reallıqda ev qiymətlərində qeyri-xətti qanunauyğunluqlar var (məsələn, OverallQual-ın qiymətə təsiri xətti deyil, eksponensialdır).
- **Çarpaz yoxlama yox idi**: Tək 80/20 bölgüsü etibarsızdır. Bir uğursuz bölgü yanıldıcı dərəcədə yaxşı və ya pis nəticə verə bilər.
- **Modellərin birləşdirilməsi (blending) yox idi**: Fərqli modellər fərqli səhvlər edir. Bir neçə modelin proqnozlarının ortalamasını almaq fərdi xətaları ləğv edir.

## TODO Yoxlama Siyahısı

- [ ] Tapşırıq 1: Kitabxanaları idxal edin
- [ ] Tapşırıq 2: Verilənləri yükləyin və hədəfi loqarifmik transformasiya edin
- [ ] Tapşırıq 3: Xüsusiyyətləri modelləşdirmə üçün ön işləyin
- [ ] Tapşırıq 4: Çarpaz yoxlama çərçivəsini qurun
- [ ] Tapşırıq 5: Ridge, Lasso və ElasticNet modellərini öyrədin
- [ ] Tapşırıq 6: Random Forest-i anlayın və öyrədin
- [ ] Tapşırıq 7: XGBoost-u anlayın və öyrədin
- [ ] Tapşırıq 8: LightGBM-i anlayın və öyrədin
- [ ] Tapşırıq 9: Bütün modelləri müqayisə edin
- [ ] Tapşırıq 10: Ən yaxşı modelləri birləşdirin (blend)
- [ ] Tapşırıq 11: Kaggle təqdimatı yaradın
- [ ] Tapşırıq 12: Xülasə və refleksiya

## Təlimatlar

Mərhələ 5-dəki nəticənizi keçmək üçün aşağıdakı qabaqcıl modelləşdirmə üsullarını tətbiq edin. Hər tapşırığı addım-addım izləyin və TODO bölmələrini öz kodunuzla doldurun.

---
## Xətti Reqressiya Bu Problem üçün Niyə Məhduddur

Xətti Reqressiya bu fərziyyələri irəli sürür:

1. **Xəttilik**: Hər xüsusiyyət ilə qiymət arasındakı əlaqə düz xəttdir. Lakin OverallQual-a baxın: keyfiyyətin 8→10 artması qiyməti 2→4 artmasından qat-qat çox yüksəldir. Bu, Xətti Reqressiyanın tuta bilmədiyi qeyri-xətti (eksponensial) təsirdir.

2. **Multikollinearlığın olmaması**: Xüsusiyyətlər bir-birindən asılı olmamalıdır. Lakin `TotalSF`, `GrLivArea` və `1stFlrSF` hamısı bir-biri ilə yüksək korrelyasiya edir. Kodlaşdırmadan sonra 200+ xüsusiyyətlə çoxu artıq (redundant) olur və bu, əmsalların qeyri-sabit və nəhəng olmasına səbəb olur.

3. **Xüsusiyyətlərə bərabər münasibət**: Xətti Reqressiya hər xüsusiyyətə cəza olmadan əmsal verir. Yüzlərlə xüsusiyyət olduqda, təlim verilənlərindəki səs-küyü (noise) əzbərləyərək overfitting edir.

### Bunun əvəzinə nəyə ehtiyacımız var:

| Problem | Həll | Model |
|---|---|---|
| Əmsallar çox böyükdür / overfitting | Əmsalları kiçiltmək üçün cəza əlavə et | **Ridge, Lasso, ElasticNet** |
| Çoxlu faydasız xüsusiyyət | Zəif xüsusiyyətləri avtomatik sıfırla | **Lasso** |
| Qeyri-xətti əlaqələr | Verilənləri bölgələrə ayıran ağaclardan istifadə et | **Random Forest, XGBoost, LightGBM** |
| Tək modelin kor nöqtələri var | Bir neçə modeli birləşdir | **Ansambl / Blending** |


---
## Tapşırıq 1: Tələb Olunan Kitabxanaların İdxalı

In [ ]:
# TODO: Əsas kitabxanaları idxal edin
# - import pandas as pd
# - import numpy as np
# - import matplotlib.pyplot as plt
# - import seaborn as sns
# - import warnings
# - warnings.filterwarnings('ignore')

# TODO: sklearn alətlərini idxal edin
# - from sklearn.model_selection import train_test_split, cross_val_score, KFold
# - from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
# - from sklearn.preprocessing import StandardScaler

# TODO: Reqlyarizasiyalı xətti modelləri idxal edin
# - from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression

# TODO: Ağac əsaslı modelləri idxal edin
# - from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
# - from xgboost import XGBRegressor
# - from lightgbm import LGBMRegressor

# TODO: Vizuallaşdırma üslubunu təyin edin
# - sns.set_style('whitegrid')
# - plt.rcParams['figure.figsize'] = (12, 6)

# TODO: Uğur mesajını çap edin


---
## Tapşırıq 2: Verilənlərin Yüklənməsi və Hədəfin Loqarifmik Transformasiyası

### SalePrice niyə loqarifmik transformasiya olunur?

Kaggle bu müsabiqəni **RMSLE** (Root Mean Squared **Log** Error) ilə qiymətləndirir:

$$\text{RMSLE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} \left( \log(\hat{y}_i + 1) - \log(y_i + 1) \right)^2}$$

Əgər təlimdən **əvvəl** hədəfimizə `log(1 + y)` tətbiq etsək, RMSLE sadə RMSE-yə çevrilir — modellərimizin məhz minimuma endirdiyi metrikə. Bu, pulsuz yaxşılaşmadır.

Bundan əlavə, SalePrice sağa əyridir (evlərin əksəriyyəti ucuzdur, bir neçəsi çox bahalıdır). Loqarifmik transformasiya paylanmanı daha normal edir və bu, xətti modellərə kömək edir.

**Vacib:** Proqnozlaşdırdıqdan sonra, faktiki dollar qiymətlərini almaq üçün bunu `np.expm1()` ilə (`exp(x) - 1` hesablayır) geri çevirməliyik.


In [ ]:
# TODO: Xüsusiyyət mühəndisliyi aparılmış təlim verilənlərini yükləyin
# - 'Solutions/train_feature_engineered.csv' faylını yükləmək üçün pd.read_csv() istifadə edin
# - 'df' dəyişənində saxlayın
# - Ölçünü (shape) çap edin

# TODO: Xüsusiyyətləri və hədəfi ayırın
# - X = df.drop('SalePrice', axis=1)
# - y_raw = df['SalePrice']  (sonradan müqayisə üçün xam qiymətləri saxlayın)

# TODO: Hədəfi loqarifmik transformasiya edin
# - y = np.log1p(y_raw)  (bu, log(1 + qiymət) hesablayır)
# - Fərqi görmək üçün y_raw.describe() ilə y.describe()-ı çap edin

# TODO: Transformasiyanı vizuallaşdırın
# - 1x2 subplot yaradın
# - Sol: y_raw histoqramı, başlıq 'Original SalePrice'
# - Sağ: y histoqramı, başlıq 'Log-Transformed SalePrice'
# - Diqqət edin, sağ qrafik daha simmetrikdir (normal formalıdır)


---
## Tapşırıq 3: Xüsusiyyətlərin Modelləşdirmə üçün Ön İşlənməsi

Modelləri öyrətməzdən əvvəl bizə lazımdır:
1. Qalan kateqorial sütunları idarə etmək (onları one-hot kodlaşdırmaq)
2. `Id` sütununu silmək (o, xüsusiyyət deyil)
3. Çatışmayan dəyərləri doldurmaq
4. Reqlyarizasiyalı xətti modellər üçün xüsusiyyətləri miqyaslamaq


In [ ]:
# TODO: Id sütunu varsa, silin
# - if 'Id' in X.columns: X = X.drop('Id', axis=1)

# TODO: Kateqorial sütunları müəyyən edin
# - cat_cols = X.select_dtypes(include=['object']).columns.tolist()
# - Neçə kateqorial sütunun qaldığını çap edin
# - Onların adlarını çap edin

# TODO: Kateqorial sütunları one-hot kodlaşdırın
# - X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
# - Kodlaşdırmadan sonra yeni ölçünü çap edin

# TODO: Qalan çatışmayan dəyərləri 0 ilə doldurun
# - X = X.fillna(0)
# - Yoxlayın: X.isnull().sum().sum() çap edin (0 olmalıdır)

# TODO: Yekun xüsusiyyət sayını çap edin


In [ ]:
# TODO: Verilənləri təlim və test dəstlərinə bölün
# - İstifadə edin: X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# - Dörd dəstin ölçülərini çap edin

# TODO: Xüsusiyyətləri miqyaslayın (Ridge, Lasso, ElasticNet üçün lazımdır)
# - scaler = StandardScaler()
# - X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
# - X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)
# - Qeyd: təlim verilənləri üzərində fit edin, test verilənləri üzərində transform edin (heç vaxt test üzərində fit etməyin!)
# - Qeyd: ağac əsaslı modellər (RF, XGBoost, LightGBM) miqyaslamaya ehtiyac DUYMUR


---
## Tapşırıq 4: Çarpaz Yoxlama (Cross-Validation) Çərçivəsinin Qurulması

### Tək train/test bölgüsü əvəzinə niyə çarpaz yoxlama?

Tək 80/20 bölgüsü tələbəni yalnız bir imtahanla qiymətləndirmək kimidir — ona şans gələ də bilər, gəlməyə də. **5-Qat Çarpaz Yoxlama (5-Fold Cross-Validation)** verilənləri 5 hissəyə bölür, 5 dəfə öyrədir (hər dəfə fərqli hissəni kənarda saxlayır) və nəticələrin ortalamasını alır. Bu, modelin Kaggle-da necə performans göstərəcəyinə dair daha etibarlı qiymətləndirmə verir.

```
Qat 1:   [TEST] [Təlim] [Təlim] [Təlim] [Təlim]  → Nəticə 1
Qat 2:   [Təlim] [TEST] [Təlim] [Təlim] [Təlim]  → Nəticə 2
Qat 3:   [Təlim] [Təlim] [TEST] [Təlim] [Təlim]  → Nəticə 3
Qat 4:   [Təlim] [Təlim] [Təlim] [TEST] [Təlim]  → Nəticə 4
Qat 5:   [Təlim] [Təlim] [Təlim] [Təlim] [TEST]  → Nəticə 5

Yekun Nəticə = Ortalama(Nəticə 1..5)  ±  Std(Nəticə 1..5)
```


In [ ]:
# TODO: K-Fold çarpaz yoxlamanı qurun
# - kf = KFold(n_splits=5, shuffle=True, random_state=42)

# TODO: Modelləri çarpaz yoxlama ilə qiymətləndirmək üçün köməkçi funksiya yaradın
# def rmse_cv(model, X, y):
#     scores = cross_val_score(model, X, y, scoring='neg_root_mean_squared_error', cv=kf)
#     return -scores  (mənfi işarəni çevirin, çünki sklearn mənfi RMSE qaytarır)
#
# İstifadə: scores = rmse_cv(model, X_train_scaled, y_train)
#        print(f"RMSE: {scores.mean():.4f} (+/- {scores.std():.4f})")

# TODO: Bütün modellərin nəticələrini saxlamaq üçün lüğət (dictionary) yaradın
# - results = {}  (hər modeli öyrətdikcə bunu dolduracağıq)

# TODO: Təsdiq mesajı çap edin


---
## Tapşırıq 5: Reqlyarizasiyalı Xətti Modellər — Ridge, Lasso, ElasticNet

### Adi Xətti Reqressiyada nə səhv getdi?

Xətti Reqressiya yalnız proqnoz xətasını (MSE) minimuma endirir. 200+ xüsusiyyətlə onun çox azadlığı var — səs-küylü xüsusiyyətlərə böyük əmsallar verir və overfitting edir.

**Reqlyarizasiya (Regularization)** modeli əmsalları kiçik saxlamağa məcbur edən cərimə həddi əlavə edir:

### Ridge (L2 Reqlyarizasiyası)

$$\text{Loss} = \text{MSE} + \alpha \sum_{j=1}^{p} \beta_j^2$$

- Cərimə olaraq **əmsalların kvadratlarının cəmini** əlavə edir
- Bütün əmsalları sıfıra doğru kiçildir, lakin heç vaxt tam sıfır etmir
- Çoxlu xüsusiyyətin hər biri bir az töhfə verdikdə yaxşıdır
- `alpha` cərimənin gücünü idarə edir: alpha nə qədər yüksəkdirsə, əmsallar bir o qədər kiçikdir

### Lasso (L1 Reqlyarizasiyası)

$$\text{Loss} = \text{MSE} + \alpha \sum_{j=1}^{p} |\beta_j|$$

- Cərimə olaraq **əmsalların mütləq qiymətlərinin cəmini** əlavə edir
- Əmsalları tam sıfıra qədər kiçildə bilər — avtomatik xüsusiyyət seçimi
- Yalnız bir neçə xüsusiyyət həqiqətən vacib olduqda yaxşıdır
- 200+ xüsusiyyətlə Lasso, çox güman ki, onların 100+-ını sıfırlayacaq

### ElasticNet (L1 + L2 Birləşməsi)

$$\text{Loss} = \text{MSE} + \alpha \left( r \sum |\beta_j| + \frac{1-r}{2} \sum \beta_j^2 \right)$$

- Ridge və Lasso-nu birləşdirir: `l1_ratio` (r) qarışığı idarə edir
- `l1_ratio=1.0` → saf Lasso, `l1_ratio=0.0` → saf Ridge
- Xüsusiyyətlər bir-biri ilə korrelyasiya etdikdə yaxşıdır (Lasso tək başına korrelyasiya edən qrupdan təsadüfi birini seçə bilər)


In [ ]:
# ===========================================================
# Addım 5.1: Baza Xətti — Adi Xətti Reqressiya (Mərhələ 5-dəki yanaşmamız)
# ===========================================================

# TODO: Adi Xətti Reqressiya modelini öyrədin
# - lr = LinearRegression()
# - scores = rmse_cv(lr, X_train_scaled, y_train)
# - results['LinearRegression'] = scores
# - Çap edin: f"Linear Regression RMSE: {scores.mean():.4f} (+/- {scores.std():.4f})"
# - Bu, keçməli olduğumuz baza xəttidir (baseline)
# - 


In [ ]:
# ===========================================================
# Addım 5.2: Ridge Reqressiyası
# ===========================================================

# TODO: Ən yaxşısını tapmaq üçün Ridge-i müxtəlif alpha dəyərləri ilə öyrədin
# - alphas = [0.1, 1.0, 5.0, 10.0, 50.0, 100.0]
# - Hər alpha üçün:
#     ridge = Ridge(alpha=alpha)
#     scores = rmse_cv(ridge, X_train_scaled, y_train)
#     print(f"  alpha={alpha:>6}: RMSE = {scores.mean():.4f} (+/- {scores.std():.4f})")

# TODO: Ridge-i ən yaxşı alpha ilə öyrədin
# - best_ridge = Ridge(alpha=<best_alpha>)
# - scores = rmse_cv(best_ridge, X_train_scaled, y_train)
# - results['Ridge'] = scores
# - Ən yaxşı nəticəni çap edin


In [ ]:
# ===========================================================
# Addım 5.3: Lasso Reqressiyası
# ===========================================================

# TODO: Lasso-nu müxtəlif alpha dəyərləri ilə öyrədin
# - alphas = [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05]
# - Qeyd: Lasso, Ridge-dən daha kiçik alpha dəyərləri istifadə edir
# - Hər alpha üçün:
#     lasso = Lasso(alpha=alpha, max_iter=10000)
#     scores = rmse_cv(lasso, X_train_scaled, y_train)
#     print(f"  alpha={alpha}: RMSE = {scores.mean():.4f} (+/- {scores.std():.4f})")

# TODO: Lasso-nu ən yaxşı alpha ilə öyrədin
# - best_lasso = Lasso(alpha=<best_alpha>, max_iter=10000)
# - scores = rmse_cv(best_lasso, X_train_scaled, y_train)
# - results['Lasso'] = scores
# - Ən yaxşı nəticəni çap edin

# TODO: Lasso-nun neçə xüsusiyyəti sıfırladığını yoxlayın
# - best_lasso.fit(X_train_scaled, y_train)
# - n_zero = (best_lasso.coef_ == 0).sum()
# - print(f"Lasso zeroed out {n_zero} of {len(best_lasso.coef_)} features")
# - Bu, avtomatik xüsusiyyət seçimidir


In [ ]:
# ===========================================================
# Addım 5.4: ElasticNet
# ===========================================================

# TODO: ElasticNet-i müxtəlif alpha və l1_ratio kombinasiyaları ilə öyrədin
# - alphas = [0.0001, 0.0005, 0.001, 0.005]
# - l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]
# - Hər kombinasiya üçün:
#     enet = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=10000)
#     scores = rmse_cv(enet, X_train_scaled, y_train)
#     Nəticələri saxlayın və çap edin

# TODO: ElasticNet-i ən yaxşı alpha və l1_ratio ilə öyrədin
# - best_enet = ElasticNet(alpha=<best_alpha>, l1_ratio=<best_l1_ratio>, max_iter=10000)
# - scores = rmse_cv(best_enet, X_train_scaled, y_train)
# - results['ElasticNet'] = scores
# - Ən yaxşı nəticəni çap edin


---
## Tapşırıq 6: Random Forest

### Niyə ağac əsaslı modellər?

Xətti modellər verilənlərin içindən düz xətt (və ya hipermüstəvi) çəkir. Lakin ev qiymətləri düz xətlərə tabe deyil:
- OverallQual=10 olan ev OverallQual=5 olan evdən sadəcə "iki dəfə bahalı" deyil — 4 dəfə bahalı ola bilər
- 2 yataq otağı və 3 hamamı olan evin qiyməti "yataq otağı təsiri" + "hamam təsiri" cəmindən fərqlidir

**Qərar Ağacları (Decision Trees)** bunu verilənləri bölgələrə ayırmaqla həll edir:
```
ƏGƏR OverallQual >= 7 VƏ GrLivArea >= 2000:
    $350,000 proqnozlaşdır (oxşar evlərin ortalaması)
ƏKS HALDA ƏGƏR OverallQual >= 7 VƏ GrLivArea < 2000:
    $250,000 proqnozlaşdır
ƏKS HALDA:
    $150,000 proqnozlaşdır
```

**Random Forest** hər biri təsadüfi verilən və xüsusiyyət alt çoxluğu üzərində olmaqla çoxlu ağac (məsələn, 500) öyrədir, sonra onların proqnozlarının ortalamasını alır. Bu, overfitting-i kəskin şəkildə azaldır.

### Əsas parametrlər:

- **`n_estimators`**: Ağacların sayı. Daha çox ağac = daha sabit proqnozlar (lakin daha yavaş). 500-1000 tipik dəyərdir.
- **`max_depth`**: Hər ağacın nə qədər dərinə böyüyə biləcəyi. Daha dərin = daha mürəkkəb qanunauyğunluqlar, lakin overfitting riski. `None` ağacların tam böyüməsinə imkan verir.
- **`min_samples_leaf`**: Yarpaq düyünündə (leaf node) minimum nümunə sayı. Daha yüksək = daha mühafizəkar ağaclar.
- **`max_features`**: Hər ağacın hər bölünmədə neçə xüsusiyyəti nəzərə aldığı. `'sqrt'` √(ümumi xüsusiyyətlər) deməkdir — müxtəlifliyi təmin edir.

**Qeyd:** Random Forest xüsusiyyətlərin miqyaslanmasına ehtiyac DUYMUR. Ağaclar hədd dəyərləri (threshold) üzrə bölür, ona görə dəyərlərin böyüklüyünün əhəmiyyəti yoxdur.


In [ ]:
# TODO: Random Forest modelini öyrədin
# - rf = RandomForestRegressor(
#       n_estimators=500,      # 500 ağac
#       max_depth=None,        # ağaclar tam böyüsün
#       min_samples_leaf=2,    # hər yarpaqda ən azı 2 nümunə
#       max_features='sqrt',   # hər bölünmədə sqrt(n_features) istifadə et
#       random_state=42,
#       n_jobs=-1              # sürət üçün bütün CPU nüvələrindən istifadə et
#   )

# TODO: Çarpaz yoxlama ilə qiymətləndirin
# - Qeyd: X_train istifadə edin (miqyaslanmamış!) — ağaclar miqyaslamaya ehtiyac duymur
# - scores = rmse_cv(rf, X_train, y_train)
# - results['RandomForest'] = scores
# - Çap edin: f"Random Forest RMSE: {scores.mean():.4f} (+/- {scores.std():.4f})"


---
## Tapşırıq 7: XGBoost (Extreme Gradient Boosting)

### XGBoost Random Forest-dən nə ilə fərqlənir?

**Random Forest**: Ağacları paralel şəkildə **müstəqil** öyrədir, sonra onların ortalamasını alır.

**XGBoost**: Ağacları **ardıcıl** öyrədir — hər yeni ağac əvvəlki bütün ağacların səhvlərini düzəltməyə diqqət yetirir.

Bunu belə təsəvvür edin:
1. Ağac 1 ev qiymətlərini proqnozlaşdırır → bəzilərini yanlış tapır
2. Ağac 2, Ağac 1-in **xətaları** üzərində öyrədilir → bəzi səhvləri düzəldir
3. Ağac 3 **qalan xətalar** üzərində öyrədilir → daha çoxunu düzəldir
4. 3000 raunddan sonra birləşmiş proqnoz çox dəqiq olur

Bu, AdaBoost ilə eyni boosting ideyasıdır (Mərhələ 6 Ansambl Öyrənməsindən), lakin XGBoost daha ağıllı riyaziyyatdan (ikinci dərəcəli qradiyent optimallaşdırması) istifadə edir və daha sürətlidir.

### Əsas parametrlər:

- **`n_estimators`**: Boosting raundlarının (ağacların) sayı. Daha çox raund = daha yaxşı, lakin overfitting-in qarşısını almaq üçün `early_stopping` istifadə edin.
- **`learning_rate`**: Hər ağacın nə qədər töhfə verdiyi. Daha aşağı (0.01-0.05) = daha çox ağac lazımdır, lakin daha yaxşı ümumiləşdirir.
- **`max_depth`**: Ağacın dərinliyi. Boosting Random Forest-dən daha dayaz ağaclar (3-6) istifadə edir.
- **`subsample`**: Hər ağac üçün istifadə olunan verilənlərin payı (0.7 = 70%). Təsadüfilik overfitting-in qarşısını alır.
- **`colsample_bytree`**: Hər ağac üçün istifadə olunan xüsusiyyətlərin payı. Random Forest-in `max_features` parametri ilə eyni ideyadır.
- **`reg_alpha`** (L1) və **`reg_lambda`** (L2): Yarpaq çəkilərinə reqlyarizasiya — Lasso/Ridge ilə eyni ideya, lakin hər ağacın daxilində tətbiq olunur.


In [ ]:
# TODO: XGBoost modelini öyrədin
# - xgb = XGBRegressor(
#       n_estimators=3000,          # 3000-ə qədər boosting raundu
#       learning_rate=0.05,         # diqqətli öyrənmə üçün kiçik öyrənmə sürəti
#       max_depth=4,                # orta dərəcədə dayaz ağaclar
#       subsample=0.7,              # hər ağac üçün verilənlərin 70%-ni istifadə et
#       colsample_bytree=0.7,       # hər ağac üçün xüsusiyyətlərin 70%-ni istifadə et
#       reg_alpha=0.1,              # L1 reqlyarizasiyası
#       reg_lambda=1.0,             # L2 reqlyarizasiyası
#       random_state=42,
#       n_jobs=-1,
#       verbosity=0                 # təlim çıxışını gizlət
#   )

# TODO: Çarpaz yoxlama ilə qiymətləndirin
# - Qeyd: X_train istifadə edin (miqyaslanmamış!) — ağaclar miqyaslamaya ehtiyac duymur
# - scores = rmse_cv(xgb, X_train, y_train)
# - results['XGBoost'] = scores
# - Çap edin: f"XGBoost RMSE: {scores.mean():.4f} (+/- {scores.std():.4f})"


---
## Tapşırıq 8: LightGBM (Light Gradient Boosting Machine)

### LightGBM XGBoost-dan nə ilə fərqlənir?

Hər ikisi qradiyent boosting üsuludur, lakin LightGBM iki əsas yenilikdən istifadə edir:

1. **Yarpaq üzrə böyümə (leaf-wise)** (XGBoost-un səviyyə üzrə - level-wise - böyüməsinə qarşı): Bütün yarpaqları eyni dərinlikdə böyütmək əvəzinə, LightGBM xətanı ən çox azaldan yarpağı böyüdür. Bu, daha az yarpaqla daha yaxşı dəqiqliyə çatır.

```
XGBoost (level-wise):         LightGBM (leaf-wise):
       O                            O
      / \                          / \
     O   O     (hamısı eyni dərinlikdə)   O   O
    / \  / \                     / \
   O  O O  O                    O   O   (ən çox fayda verdiyi yerdə ən dərinə böyüyür)
```

2. **Histoqram əsaslı bölünmə**: Hər mümkün bölünmə nöqtəsini yoxlamaq əvəzinə, LightGBM xüsusiyyət dəyərlərini qutulara (histoqramlara) qruplaşdırır. Bu, onu böyük verilənlər dəstlərində XGBoost-dan 10-20 dəfə sürətli edir.

### Əsas parametrlər:

- **`n_estimators`**: Boosting raundlarının sayı.
- **`learning_rate`**: Addım ölçüsünün kiçildilməsi. XGBoost-dakı ilə eyni rol.
- **`num_leaves`**: Ağac başına maksimum yarpaq sayı. Mürəkkəbliyi idarə edir (defolt: 31). Daha çox yarpaq = daha mürəkkəb.
- **`max_depth`**: Maksimum ağac dərinliyi. Məhdudiyyət olmaması üçün -1 təyin edin (əvəzinə `num_leaves` ilə idarə olunur).
- **`subsample`** və **`colsample_bytree`**: XGBoost-dakı ilə eyni təsadüfilik idarəetmələri.
- **`reg_alpha`** və **`reg_lambda`**: L1/L2 reqlyarizasiyası.


In [ ]:
# TODO: LightGBM modelini öyrədin
# - lgbm = LGBMRegressor(
#       n_estimators=3000,          # 3000-ə qədər boosting raundu
#       learning_rate=0.05,         # kiçik öyrənmə sürəti
#       num_leaves=31,              # defolt, ağacın mürəkkəbliyini idarə edir
#       max_depth=4,                # dərinliyi məhdudlaşdır
#       subsample=0.7,              # hər ağac üçün verilənlərin 70%-ni istifadə et
#       colsample_bytree=0.7,       # hər ağac üçün xüsusiyyətlərin 70%-ni istifadə et
#       reg_alpha=0.1,              # L1 reqlyarizasiyası
#       reg_lambda=1.0,             # L2 reqlyarizasiyası
#       random_state=42,
#       n_jobs=-1,
#       verbosity=-1                # təlim çıxışını gizlət
#   )

# TODO: Çarpaz yoxlama ilə qiymətləndirin
# - Qeyd: X_train istifadə edin (miqyaslanmamış!) — ağaclar miqyaslamaya ehtiyac duymur
# - scores = rmse_cv(lgbm, X_train, y_train)
# - results['LightGBM'] = scores
# - Çap edin: f"LightGBM RMSE: {scores.mean():.4f} (+/- {scores.std():.4f})"


---
## Tapşırıq 9: Bütün Modellərin Müqayisəsi

İndi görək hansı modellər ən yaxşı nəticə göstərdi və niyə.

In [ ]:
# TODO: Müqayisə cədvəli yaradın
# - comparison = pd.DataFrame({
#       'Model': results.keys(),
#       'Mean RMSE': [scores.mean() for scores in results.values()],
#       'Std RMSE': [scores.std() for scores in results.values()]
#   })
# - comparison = comparison.sort_values('Mean RMSE')
# - Müqayisə cədvəlini çap edin

# TODO: Modellərin müqayisəsini vizuallaşdırın
# - x oxunda model adları, y oxunda Mean RMSE olan bar plot yaradın
# - Std RMSE istifadə edərək xəta çubuqları (error bars) əlavə edin
# - Ən yaxşı modeli fərqli rənglə göstərin
# - Başlıq əlavə edin: 'Model Comparison (5-Fold CV RMSE on Log-Transformed Target)'
# - Daha aşağı RMSE = daha yaxşı

# TODO: Qalibi çap edin
# - Ən aşağı Mean RMSE-yə malik modeli çap edin


In [ ]:
# TODO: Bütün modelləri tam təlim dəsti üzərində öyrədin və ayrılmış test dəstində qiymətləndirin
#
# Hər model üçün test dəstindəki faktiki RMSE-ni görmək istəyirik:

# - Xətti modellər üçün (miqyaslanmış verilənlərdən istifadə edin): X_train_scaled, X_test_scaled
# - Ağac modelləri üçün (miqyaslanmamış verilənlərdən istifadə edin): X_train, X_test

# TODO: Hər model üçün:
# - model.fit(X_train_appropriate, y_train)
# - y_pred = model.predict(X_test_appropriate)
# - rmse = np.sqrt(mean_squared_error(y_test, y_pred))
# - r2 = r2_score(y_test, y_pred)
# - Modelin adını, RMSE və R² qiymətlərini çap edin

# TODO: Ən yaxşı model üçün Faktiki və Proqnozlaşdırılan scatter plot yaradın
# - Proqnozları dollar qiymətlərinə çevirin: np.expm1(y_pred)
# - Faktiki dəyərləri geri çevirin: np.expm1(y_test)
# - Diaqonal xətli scatter plot
# - Başlıqda modelin adı və R² qiyməti


---
## Tapşırıq 10: Ən Yaxşı Modellərin Birləşdirilməsi (Blending)

### Blending niyə işləyir

Fərqli modellərin fərqli güclü tərəfləri var:
- **Lasso/Ridge**: Xətti meylləri tutmaqda yaxşıdır, görünməmiş verilənlərdə sabitdir
- **XGBoost/LightGBM**: Qeyri-xətti qanunauyğunluqları və qarşılıqlı təsirləri tutmaqda yaxşıdır

Onların proqnozlarının ortalamasını aldıqda, bir modelin xətaları digəri tərəfindən qismən ləğv olunur. Bu, demək olar ki, həmişə nəticəni yaxşılaşdırır.

**Sadə çəkili ortalama (weighted average)** ən asan və ən effektiv yanaşmadır:

$$\hat{y}_{\text{blend}} = w_1 \cdot \hat{y}_{\text{Ridge}} + w_2 \cdot \hat{y}_{\text{Lasso}} + w_3 \cdot \hat{y}_{\text{XGBoost}} + w_4 \cdot \hat{y}_{\text{LightGBM}}$$

burada $w_1 + w_2 + w_3 + w_4 = 1$

**İpucu:** Daha aşağı CV RMSE-yə malik modellərə daha çox çəki verin.


In [ ]:
# TODO: Bütün modelləri tam təlim dəsti üzərində öyrədin
# - best_ridge.fit(X_train_scaled, y_train)
# - best_lasso.fit(X_train_scaled, y_train)
# - xgb.fit(X_train, y_train)
# - lgbm.fit(X_train, y_train)

# TODO: Test dəstində hər modeldən proqnozlar əldə edin
# - pred_ridge = best_ridge.predict(X_test_scaled)
# - pred_lasso = best_lasso.predict(X_test_scaled)
# - pred_xgb = xgb.predict(X_test)
# - pred_lgbm = lgbm.predict(X_test)

# TODO: Proqnozları çəkilərlə birləşdirin
# - Bərabər çəkilərlə başlayın: hər biri 0.25
# - pred_blend = 0.25 * pred_ridge + 0.25 * pred_lasso + 0.25 * pred_xgb + 0.25 * pred_lgbm

# TODO: Birləşdirilmiş proqnozu qiymətləndirin
# - rmse_blend = np.sqrt(mean_squared_error(y_test, pred_blend))
# - r2_blend = r2_score(y_test, pred_blend)
# - Çap edin: f"Blended RMSE: {rmse_blend:.4f}, R²: {r2_blend:.4f}"
# - Fərdi modellərin nəticələri ilə müqayisə edin


In [ ]:
# TODO: Müxtəlif blending çəkiləri ilə təcrübə edin
# - Ən yaxşı performans göstərən modellərə daha çox çəki verməyə çalışın
# - Nümunə: 0.15 * ridge + 0.15 * lasso + 0.35 * xgb + 0.35 * lgbm

# TODO: Bir neçə çəki kombinasiyasını sınayın və birləşdirilmiş RMSE-ni izləyin
# - weight_configs = [
#     (0.25, 0.25, 0.25, 0.25),  # bərabər
#     (0.15, 0.15, 0.35, 0.35),  # ağac modellərinə üstünlük ver
#     (0.10, 0.10, 0.40, 0.40),  # ağaclara güclü üstünlük ver
#     (0.20, 0.20, 0.30, 0.30),  # ağaclara cüzi üstünlük
#   ]
# - Hər (w1, w2, w3, w4) üçün:
#     pred = w1*pred_ridge + w2*pred_lasso + w3*pred_xgb + w4*pred_lgbm
#     rmse = np.sqrt(mean_squared_error(y_test, pred))
#     print(f"Weights ({w1},{w2},{w3},{w4}): RMSE = {rmse:.4f}")

# TODO: Ən yaxşı çəki kombinasiyasını seçin və saxlayın
# - best_weights = (<w1>, <w2>, <w3>, <w4>)


---
## Tapşırıq 11: Kaggle Təqdimatının Yaradılması

İndi Kaggle-ın gözlədiyi formatda real təqdimat faylı hazırlayaq:
```
Id,SalePrice
1461,169000.1
1462,187724.1
...
```

**Kritik addımlar:**
1. Test dəstinə tam eyni ön işləməni tətbiq edin
2. Proqnozları loqarifmik məkanda verin
3. Dollar qiymətlərini almaq üçün `np.expm1()` ilə tərs transformasiya edin
4. Mənfi proqnozları 0-a qədər kəsin (qiymətlər mənfi ola bilməz)


In [ ]:
# TODO: Test verilənlərini yükləyin
# - test_df = pd.read_csv('Solutions/test_feature_engineered.csv')
# - test_ids = test_df['Id']  (təqdimat üçün ID-ləri saxlayın)
# - X_kaggle = test_df.drop('Id', axis=1, errors='ignore')
# - test_df.shape çap edin

# TODO: Təlimdəki ilə eyni ön işləməni tətbiq edin
# - Eyni sütunlarla one-hot kodlaşdırın:
#   X_kaggle = pd.get_dummies(X_kaggle, columns=cat_cols_from_training, drop_first=True)
# - Sütunları təlim dəsti ilə uyğunlaşdırın (çatışmayan sütunları 0 kimi əlavə edin, artıqları silin):
#   X_kaggle = X_kaggle.reindex(columns=X_train.columns, fill_value=0)
# - Çatışmayan dəyərləri doldurun: X_kaggle = X_kaggle.fillna(0)
# - Xətti modellər üçün miqyaslayın: X_kaggle_scaled = pd.DataFrame(scaler.transform(X_kaggle), columns=X_kaggle.columns)

# TODO: Uyğunluğu yoxlamaq üçün ölçüləri çap edin
# - print(f"Training features: {X_train.shape[1]}")
# - print(f"Test features:     {X_kaggle.shape[1]}")
# - Bunlar üst-üstə düşməlidir!


In [ ]:
# TODO: Kaggle test dəstində birləşdirilmiş (blended) proqnozlar yaradın
# - Tapşırıq 10-dan ən yaxşı çəkilərdən istifadə edin
# - (w1, w2, w3, w4) = best_weights
# - pred_kaggle = (
#       w1 * best_ridge.predict(X_kaggle_scaled) +
#       w2 * best_lasso.predict(X_kaggle_scaled) +
#       w3 * xgb.predict(X_kaggle) +
#       w4 * lgbm.predict(X_kaggle)
#   )

# TODO: Faktiki qiymətləri almaq üçün loqarifmik transformasiyanı tərs çevirin
# - final_predictions = np.expm1(pred_kaggle)
# - final_predictions = np.maximum(final_predictions, 0)  (mənfi dəyərləri 0-a qədər kəsin)

# TODO: Təqdimat DataFrame-i yaradın
# - submission = pd.DataFrame({'Id': test_ids, 'SalePrice': final_predictions})
# - submission.head() çap edin
# - Qiymət aralığını yoxlamaq üçün submission.describe() çap edin

# TODO: Təqdimat faylını yadda saxlayın
# - submission.to_csv('submission_advanced.csv', index=False)
# - Təsdiq mesajını və fayl yolunu çap edin


---
## Tapşırıq 12: Xülasə və Refleksiya

### Nələri Öyrəndik

1. **SalePrice-ı loqarifmik transformasiya etmək** RMSLE-ni RMSE-yə çevirir və paylanmanı normallaşdırır
2. **Reqlyarizasiya** (Ridge, Lasso, ElasticNet) çoxlu xüsusiyyət olduqda əmsalların partlamasının qarşısını alır
3. **Lasso** əhəmiyyətsiz xüsusiyyətləri sıfırlayaraq avtomatik xüsusiyyət seçimi həyata keçirir
4. **Ağac əsaslı modellər** (Random Forest, XGBoost, LightGBM) xətti modellərin qaçırdığı qeyri-xətti qanunauyğunluqları tutur
5. **XGBoost və LightGBM** ardıcıl boosting istifadə edir — hər ağac əvvəlkinin səhvlərini düzəldir
6. **Çarpaz yoxlama** tək təsadüfi bölgüdən asılı olmayan etibarlı performans qiymətləndirməsi verir
7. **Modellərin birləşdirilməsi (blending)** fərqli modellərin güclü tərəflərini birləşdirir və demək olar ki, həmişə nəticəni yaxşılaşdırır

### Performans Sualları — Bunlara Cavab Verin:

1. **Yaxşılaşma**: Mərhələ 5-dəki Kaggle nəticəniz nə idi? Yeni nəticəniz nədir? Nə qədər yaxşılaşdı?

2. **Ən yaxşı tək model**: Çarpaz yoxlamada hansı tək model ən yaxşı nəticəni göstərdi? Sizcə niyə?

3. **Xətti və Ağac**: XGBoost/LightGBM Ridge/Lasso-dan nə qədər yaxşı idi? Bu, verilənlər haqqında sizə nə deyir?

4. **Blending faydası**: Blending ən yaxşı tək modeldən daha yaxşı nəticə verdimi? Nə qədər?

5. **Lasso ilə xüsusiyyət seçimi**: Lasso neçə xüsusiyyəti sıfırladı? Bu, verilənlər dəsti haqqında nəyi göstərir?

6. **Əsas nəticə**: Hansı vəziyyətlərdə XGBoost əvəzinə Xətti Reqressiyanı seçərdiniz? (İpucu: şərh oluna bilənlik, kiçik verilənlər dəstləri və deployment barədə düşünün)

### Təhlilinizi BURAYA yazın:

(Bunu yazılı təhlilinizlə əvəz edin)

---
## BONUS Tapşırıq 1: Ağac Modellərindən Xüsusiyyət Əhəmiyyəti

Ağac əsaslı modellər bizə hansı xüsusiyyətlərin ən vacib olduğunu deyə bilər.

In [ ]:
# TODO: XGBoost-dan xüsusiyyət əhəmiyyətini çıxarın
# - xgb.fit(X_train, y_train)  (əgər hələ öyrədilməyibsə)
# - importance = pd.DataFrame({
#       'Feature': X_train.columns,
#       'Importance': xgb.feature_importances_
#   }).sort_values('Importance', ascending=False)

# TODO: Ən vacib ilk 20 xüsusiyyəti qrafikdə göstərin
# - plt.figure(figsize=(10, 8))
# - top20 = importance.head(20)
# - plt.barh(top20['Feature'], top20['Importance'], color='teal')
# - plt.xlabel('Feature Importance')
# - plt.title('Top 20 Features (XGBoost)')
# - plt.gca().invert_yaxis()  (ən yüksək əhəmiyyət yuxarıda olsun)
# - plt.tight_layout()
# - plt.show()

# TODO: Lasso-nun xüsusiyyət seçimi ilə müqayisə edin
# - Hansı xüsusiyyətlər həm XGBoost-da, həm də Lasso-da vacib görünür?
# - Kəsişməni (overlap) çap edin


---
## BONUS Tapşırıq 2: GridSearchCV ilə Hiperparametr Tənzimlənməsi

Daha da yaxşı performans üçün ən yaxşı modeli dəqiqləşdirin.

In [ ]:
# TODO: GridSearchCV idxal edin
# - from sklearn.model_selection import GridSearchCV

# TODO: LightGBM (və ya XGBoost) üçün parametr şəbəkəsi (grid) təyin edin
# - param_grid = {
#       'n_estimators': [1000, 2000, 3000],
#       'learning_rate': [0.01, 0.05, 0.1],
#       'max_depth': [3, 4, 5],
#       'num_leaves': [15, 31, 63],
#   }
#
# Qeyd: Bu şəbəkədə 3*3*3*3 = 81 kombinasiya × 5 qat = 405 model təlimi var
# Çox yavaş olarsa, şəbəkəni kiçildin və ya əvəzinə RandomizedSearchCV istifadə edin

# TODO: GridSearchCV-ni işə salın
# - grid_search = GridSearchCV(
#       LGBMRegressor(random_state=42, n_jobs=-1, verbosity=-1),
#       param_grid,
#       cv=kf,
#       scoring='neg_root_mean_squared_error',
#       verbose=1
#   )
# - grid_search.fit(X_train, y_train)

# TODO: Ən yaxşı parametrləri və nəticəni çap edin
# - print(f"Best Parameters: {grid_search.best_params_}")
# - print(f"Best CV RMSE: {-grid_search.best_score_:.4f}")

# TODO: Tənzimlənmiş modeli blend-ə daxil edin və yenidən qiymətləndirin


---
## BONUS Tapşırıq 3: Stacking (Qabaqcıl Ansambl)

Stacking sadə ortalamadan kənara çıxır: baza model proqnozlarını birləşdirməyin optimal yolunu öyrənən **meta-model** öyrədir.

```
Baza Modellər:        Meta-Model:
Ridge    → pred_1 ─┐
Lasso    → pred_2 ─┤→ LinearRegression → yekun proqnoz
XGBoost  → pred_3 ─┤
LightGBM → pred_4 ─┘
```


In [ ]:
# TODO: StackingRegressor idxal edin
# - from sklearn.ensemble import StackingRegressor

# TODO: Stacking ansamblını yaradın
# - stacker = StackingRegressor(
#       estimators=[
#           ('ridge', best_ridge),
#           ('lasso', best_lasso),
#           ('xgb', xgb),
#           ('lgbm', lgbm)
#       ],
#       final_estimator=Ridge(alpha=10.0),  # meta-model
#       cv=kf
#   )
#
# Qeyd: stacker xətti baza modellər üçün miqyaslanmış, ağaclar üçün miqyaslanmamış verilənlərdən istifadə edir.
# StackingRegressor bütün qiymətləndiricilərə (estimators) eyni verilənləri ötürdüyü üçün,
# burada yalnız ağac modellərindən istifadə etmək və ya miqyaslama üçün Pipeline istifadə etmək istəyə bilərsiniz.
# Alternativ olaraq, Tapşırıq 10-dakı sadə blending yanaşmasından istifadə edin.

# TODO: Stacker-i qiymətləndirin
# - scores = rmse_cv(stacker, X_train, y_train)
# - print(f"Stacking RMSE: {scores.mean():.4f} (+/- {scores.std():.4f})")
# - Tapşırıq 10-dakı sadə blending ilə müqayisə edin


---
## Təqdimat Yoxlama Siyahısı

Notebook-unuzun aşağıdakıları ehtiva etdiyindən əmin olun:

- ✓ Bütün tələb olunan kitabxanalar idxal edilib (xgboost, lightgbm daxil olmaqla)
- ✓ Verilənlər loqarifmik transformasiya olunmuş hədəflə yüklənib
- ✓ Xüsusiyyətlər ön işlənib (kodlaşdırılıb, miqyaslanıb, çatışmayan dəyər yoxdur)
- ✓ Çarpaz yoxlama çərçivəsi qurulub və ardıcıl istifadə olunub
- ✓ Ridge, Lasso, ElasticNet alpha tənzimlənməsi ilə öyrədilib
- ✓ Random Forest öyrədilib
- ✓ XGBoost düzgün hiperparametrlərlə öyrədilib
- ✓ LightGBM düzgün hiperparametrlərlə öyrədilib
- ✓ Bütün modellər cədvəldə və qrafikdə müqayisə edilib
- ✓ Modellər optimallaşdırılmış çəkilərlə birləşdirilib
- ✓ Kaggle təqdimat faylı yaradılıb (submission_advanced.csv)
- ✓ Bütün refleksiya sualları cavablandırılıb
- ✓ Notebook boyunca aydın sənədləşdirmə və şərhlər mövcuddur

## Əsas Nəticələr

1. Kaggle RMSLE qiymətləndirməsindən istifadə etdikdə **hədəfin loqarifmik transformasiyası** vacibdir
2. **Reqlyarizasiya** (Ridge/Lasso) çoxlu xüsusiyyət olduqda overfitting-in qarşısını alır
3. **Lasso** həm də xüsusiyyət seçicisi kimi çıxış edir — 200+ xüsusiyyətinizdən hansıların həqiqətən vacib olduğunu deyir
4. **Ağac əsaslı modellər** (XGBoost, LightGBM) xətti modellərin əsaslı şəkildə tuta bilmədiyi qeyri-xətti qanunauyğunluqları tutur
5. **Çarpaz yoxlama** etibarsız tək bölgülü qiymətləndirməni əvəz edir
6. **Modellərin birləşdirilməsi (blending)** rəqabətli ML-də ən effektiv tək üsuldur
7. **Heç bir tək model həmişə ən yaxşı deyil** — ansamblın gücü ondadır ki, fərqli modellər bir-birinin zəifliklərini kompensasiya edir